# Notebook 03: Pilot run, timing and search budget

**Study:** predicting recorded road casualty severity (KSI) in Great Britain. This notebook implements the pilot described in sections 3.5 and 3.7.2.

**What it does**
1. Writes the shared modelling code to Drive (`code/stats19_lib.py`) so notebooks 03 and 04 use identical pipelines.
2. Checks the saved folds are the same file as at the last run (checksum).
3. Fits each of the nine model and training combinations once on the fixed 20% pilot sample and records time, memory and size of the encoded data.
4. Projects the cost of the full grouped search and **sets the search budget**. Each model is tuned once, on the untreated fit (original class proportions); the chosen settings are then reused for the weighted and oversampled fits, so only the training treatment differs between them.
5. Saves the budget, the search space and the timing table.

**What it does not do:** no tuning decisions, no model choice and no use of 2025. The pilot scores are only a check that each pipeline runs and beats the base rate.

Runtime: standard CPU. If a model runs out of memory, switch High RAM on and rerun this notebook.

In [1]:
# 1. Setup -----------------------------------------------------------------
import os, sys, json, hashlib, subprocess, time, resource, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
try:
    import imblearn, psutil
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn', 'psutil'], check=True)
    import imblearn, psutil
try:
    display
except NameError:
    display = print

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/stats19_severity')
else:
    PROJECT = Path(os.environ.get('PROJECT_DIR', './stats19_severity'))
CODE = PROJECT / 'code'; MOD = PROJECT / 'outputs' / 'models'; ENV = PROJECT / 'environment'
for p in (CODE, MOD, ENV): p.mkdir(parents=True, exist_ok=True)

SEED = 20261107
N_JOBS = os.cpu_count()
TIME_BUDGET_HOURS = 5.0      # total compute for tuning plus the other two treatments (edit if you wish)
(ENV / 'pip_freeze_notebook03.txt').write_text(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)
print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__, '| imbalanced-learn', imblearn.__version__)
print('CPU cores:', N_JOBS, '| RAM (GB):', round(psutil.virtual_memory().total / 1e9, 1))

Mounted at /content/drive
Python 3.13.18 | pandas 2.2.3 | scikit-learn 1.6.1 | imbalanced-learn 0.14.2
CPU cores: 2 | RAM (GB): 13.6


In [2]:
# 2. Shared modelling code, saved to Drive ---------------------------------------
LIB = r'''"""Shared code for the STATS19 severity study (notebooks 03 onward).
Every learned step (medians, rare-category pooling, scaling, spline knots, oversampling)
is fitted on the training fold only, inside fit_predict()."""
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from imblearn.over_sampling import RandomOverSampler

RARE_MIN_FREQ = 100          # categories with fewer training-fold casualties are pooled into one level
RF_TREES = 100               # fixed; the number of trees is not tuned (3.7.2)
KINDS = ['lr', 'rf', 'hgb']
TREATMENTS = ['untreated', 'weight', 'oversample']

def load_dev(project):
    project = Path(project)
    summ = json.loads((project / 'outputs' / 'features' / 'preprocessing_summary.json').read_text())
    p = project / 'data' / 'model_inputs'
    f = p / 'features_dev_2023_2024.parquet'
    df = pd.read_parquet(f) if f.exists() else pd.read_pickle(p / 'features_dev_2023_2024.pkl')
    return df, summ['numeric'], summ['categorical']

def make_preprocessor(kind, num, cat):
    ohe = OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=RARE_MIN_FREQ,
                        sparse_output=(kind == 'lr'), dtype=np.float32)
    if kind == 'lr':
        age = [c for c in ('age_of_casualty',) if c in num]
        other = [c for c in num if c not in age]
        tf = []
        if age:
            tf.append(('age_spline', Pipeline([('imp', SimpleImputer(strategy='median')),
                       ('spl', SplineTransformer(n_knots=6, degree=3, knots='quantile', include_bias=False))]), age))
            tf.append(('age_missing', MissingIndicator(features='all'), age))
        if other:
            tf.append(('num', Pipeline([('imp', SimpleImputer(strategy='median', add_indicator=True)),
                                        ('sc', StandardScaler())]), other))
    elif kind == 'rf':
        tf = [('num', SimpleImputer(strategy='median', add_indicator=True), num)] if num else []
    else:  # hgb handles missing numerics natively
        tf = [('num', 'passthrough', num)] if num else []
    tf.append(('cat', ohe, cat))
    return ColumnTransformer(tf, remainder='drop', sparse_threshold=(0.3 if kind == 'lr' else 0.0),
                             verbose_feature_names_out=False)

ANCHOR = {
    'lr': dict(C=0.1),
    'rf': dict(max_features='sqrt', min_samples_leaf=20, min_samples_split=20),
    'hgb': dict(learning_rate=0.1, max_iter=200, max_leaf_nodes=31, min_samples_leaf=50, l2_regularization=1.0),
}
SPACE = {
    'lr': dict(C=[0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]),
    'rf': dict(max_features=['sqrt', 0.1, 0.2], min_samples_leaf=[5, 10, 20, 50], min_samples_split=[10, 20, 50]),
    'hgb': dict(learning_rate=[0.03, 0.05, 0.1], max_iter=[100, 200, 300, 500], max_leaf_nodes=[15, 31, 63],
                min_samples_leaf=[20, 50, 100], l2_regularization=[0.0, 1.0, 10.0]),
}

def sample_configs(kind, n, seed):
    """Anchor configuration first, then a seeded random draw without replacement from the grid."""
    import itertools
    keys = list(SPACE[kind])
    grid = [dict(zip(keys, v)) for v in itertools.product(*[SPACE[kind][k] for k in keys])]
    if kind == 'lr':
        idx = np.unique(np.linspace(0, len(grid) - 1, num=min(n, len(grid))).round().astype(int))
        return [grid[i] for i in idx]
    anchor = ANCHOR[kind]
    rest = [g for g in grid if g != anchor]
    rng = np.random.RandomState(seed)
    pick = rng.choice(len(rest), size=min(n - 1, len(rest)), replace=False)
    return [anchor] + [rest[i] for i in pick]

def make_model(kind, params, treatment, seed, n_jobs=-1):
    cw = 'balanced' if treatment == 'weight' else None
    if kind == 'lr':
        return LogisticRegression(C=params['C'], solver='lbfgs', max_iter=300, class_weight=cw, random_state=seed)
    if kind == 'rf':
        return RandomForestClassifier(n_estimators=RF_TREES, max_features=params['max_features'],
                                      min_samples_leaf=params['min_samples_leaf'], min_samples_split=params['min_samples_split'],
                                      class_weight=cw, n_jobs=n_jobs, random_state=seed)
    return HistGradientBoostingClassifier(max_iter=params['max_iter'], learning_rate=params['learning_rate'],
                                          max_leaf_nodes=params['max_leaf_nodes'], min_samples_leaf=params['min_samples_leaf'],
                                          l2_regularization=params['l2_regularization'], early_stopping=False,
                                          class_weight=cw, random_state=seed)

def _nbytes(a):
    return (a.data.nbytes + a.indices.nbytes + a.indptr.nbytes) if hasattr(a, 'indptr') else a.nbytes

def fit_predict(kind, params, treatment, Xtr, ytr, Xva, num, cat, seed, n_jobs=-1):
    t0 = time.time()
    pre = make_preprocessor(kind, num, cat)
    Atr = pre.fit_transform(Xtr[num + cat]); Ava = pre.transform(Xva[num + cat])
    if kind != 'lr':
        Atr = Atr.astype(np.float32); Ava = Ava.astype(np.float32)
    n_cols = Atr.shape[1]
    y = np.asarray(ytr)
    if treatment == 'oversample':
        Atr, y = RandomOverSampler(sampling_strategy=1.0, random_state=seed).fit_resample(Atr, y)
    t_prep = time.time() - t0
    model = make_model(kind, params, treatment, seed, n_jobs)
    t1 = time.time(); model.fit(Atr, y); t_fit = time.time() - t1
    t2 = time.time(); p = model.predict_proba(Ava)[:, 1]; t_pred = time.time() - t2
    info = dict(n_cols=int(n_cols), n_train=int(Atr.shape[0]), matrix_mb=round(_nbytes(Atr) / 1e6, 1),
                t_prep=round(t_prep, 2), t_fit=round(t_fit, 2), t_pred=round(t_pred, 2))
    return p, info

def metrics(y, p):
    return dict(ap=average_precision_score(y, p), auc=roc_auc_score(y, p), brier=brier_score_loss(y, p))
'''
(CODE / 'stats19_lib.py').write_text(LIB)
sys.path.insert(0, str(CODE))
import importlib, stats19_lib
importlib.reload(stats19_lib)
from stats19_lib import *
print('Saved', CODE / 'stats19_lib.py', '| sha256', hashlib.sha256(LIB.encode()).hexdigest()[:16])

Saved /content/drive/MyDrive/stats19_severity/code/stats19_lib.py | sha256 ab3f7748e0fc5fd1


In [3]:
# 3. Load development data and verify the saved folds -----------------------------
dev, NUM, CAT = load_dev(PROJECT)
FEATURES = NUM + CAT
print('Development rows:', len(dev), '| predictors:', len(FEATURES), f'({len(NUM)} numeric, {len(CAT)} categorical)')
assert not any('severity' in c for c in FEATURES)

folds_path = PROJECT / 'data' / 'model_inputs' / 'folds_dev.csv'
sha = hashlib.sha256(folds_path.read_bytes()).hexdigest()
rec = MOD / 'folds_checksum.json'
if rec.exists():
    old = json.loads(rec.read_text())['sha256']
    assert old == sha, 'folds_dev.csv has changed since it was first recorded. Do not continue; restore the original file.'
    print('Folds checksum matches the first record.')
else:
    rec.write_text(json.dumps(dict(sha256=sha, recorded=dt.datetime.now().isoformat(timespec='seconds'), sklearn=sklearn.__version__), indent=2))
    print('Folds checksum recorded:', sha[:16])
fl = pd.read_csv(folds_path)
assert (fl['cas_key'].values == dev['cas_key'].values).all() and (fl['fold'].values == dev['fold'].values).all()
print('Rows per fold:', dev['fold'].value_counts().sort_index().to_dict(), '| KSI rate', round(dev['ksi'].mean(), 4))

Development rows: 261249 | predictors: 47 (8 numeric, 39 categorical)
Folds checksum matches the first record.
Rows per fold: {0: 52411, 1: 52571, 2: 51958, 3: 52062, 4: 52247} | KSI rate 0.2117


In [4]:
# 4. Pilot fits: one per model and training combination ---------------------------------
# Training rows = pilot casualties outside fold 0; validation rows = pilot casualties in fold 0.
pilot = dev[dev['pilot'] == 1]
tr = pilot[pilot['fold'] != 0]; va = pilot[pilot['fold'] == 0]
print(f'Pilot training {len(tr)} rows (KSI {tr.ksi.mean():.3f}) | pilot validation {len(va)} rows (KSI {va.ksi.mean():.3f})')

rows = []
proc = psutil.Process()
for kind in KINDS:
    for trt in TREATMENTS:
        rss0 = proc.memory_info().rss
        p, info = fit_predict(kind, ANCHOR[kind], trt, tr, tr['ksi'], va, NUM, CAT, SEED, N_JOBS)
        m = metrics(va['ksi'], p)
        rss1 = proc.memory_info().rss
        rows.append(dict(model=kind, training=trt, **info, rss_gb=round(rss1 / 1e9, 2), **{k: round(v, 4) for k, v in m.items()}))
        print(f"{kind:>3} | {trt:<10} | cols {info['n_cols']:>4} | rows {info['n_train']:>6} | prep {info['t_prep']:>5}s fit {info['t_fit']:>6}s | AP {m['ap']:.3f} AUC {m['auc']:.3f}")
pilot_tbl = pd.DataFrame(rows)
pilot_tbl.to_csv(MOD / 'pilot_timing.csv', index=False)
base = va['ksi'].mean()
print(f'\nBase rate (average precision of a model with no information) = {base:.3f}')
assert (pilot_tbl['ap'] > base + 0.03).all(), 'A pipeline does not beat the base rate on the pilot: inspect before continuing.'
display(pilot_tbl[['model', 'training', 'n_cols', 'n_train', 'matrix_mb', 't_prep', 't_fit', 't_pred', 'rss_gb', 'ap', 'auc']])
print('Peak memory so far (GB):', round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2))

Pilot training 41595 rows (KSI 0.213) | pilot validation 10659 rows (KSI 0.214)
 lr | untreated  | cols  344 | rows  41595 | prep   0.7s fit    1.8s | AP 0.437 AUC 0.741
 lr | weight     | cols  344 | rows  41595 | prep  0.45s fit   1.45s | AP 0.435 AUC 0.740
 lr | oversample | cols  344 | rows  65450 | prep  0.52s fit   3.02s | AP 0.435 AUC 0.739
 rf | untreated  | cols  338 | rows  41595 | prep   0.5s fit  12.26s | AP 0.430 AUC 0.738
 rf | weight     | cols  338 | rows  41595 | prep  0.47s fit  12.22s | AP 0.430 AUC 0.740
 rf | oversample | cols  338 | rows  65450 | prep  0.59s fit  19.67s | AP 0.431 AUC 0.739
hgb | untreated  | cols  333 | rows  41595 | prep  0.48s fit  29.97s | AP 0.437 AUC 0.744
hgb | weight     | cols  333 | rows  41595 | prep  0.46s fit  30.68s | AP 0.439 AUC 0.743
hgb | oversample | cols  333 | rows  65450 | prep  0.74s fit  41.83s | AP 0.436 AUC 0.739

Base rate (average precision of a model with no information) = 0.214


,model,training,n_cols,n_train,matrix_mb,t_prep,t_fit,t_pred,rss_gb,ap,auc
0,lr,untreated,344,41595,27.0,0.70,1.80,0.00,0.58,0.4373,0.7406
1,lr,weight,344,41595,27.0,0.45,1.45,0.00,0.59,0.4346,0.7399
2,lr,oversample,344,65450,42.5,0.52,3.02,0.00,0.70,0.4352,0.7389
3,rf,untreated,338,41595,56.2,0.50,12.26,0.28,0.72,0.4300,0.7383
4,rf,weight,338,41595,56.2,0.47,12.22,0.24,0.73,0.4300,0.7398
5,rf,oversample,338,65450,88.5,0.59,19.67,0.24,0.58,0.4311,0.7391
6,hgb,untreated,333,41595,55.4,0.48,29.97,1.68,0.62,0.4369,0.7437
7,hgb,weight,333,41595,55.4,0.46,30.68,1.73,0.62,0.4394,0.7429
8,hgb,oversample,333,65450,87.2,0.74,41.83,1.52,0.62,0.4361,0.7395


Peak memory so far (GB): 0.91


In [5]:
# 5. Project the full search and set the budget ----------------------------------------
# Full training fold = four-fifths of development rows. Time is scaled by (row ratio)^1.1, a deliberately
# cautious assumption for tree models; notebook 04 reports actual against projected time for the first full fit.
n_full = int((dev['fold'] != 0).sum())
scale = n_full / len(tr)
EXP = 1.1
pilot_tbl['t_full_fit_min'] = ((pilot_tbl['t_prep'] + pilot_tbl['t_fit'] + pilot_tbl['t_pred']) * scale ** EXP / 60).round(2)
pilot_tbl['matrix_full_gb'] = (pilot_tbl['matrix_mb'] * scale / 1000).round(2)
print(f'Scale factor: {scale:.2f} (full training fold {n_full} rows vs pilot {len(tr)})')
display(pilot_tbl[['model', 'training', 't_full_fit_min', 'matrix_full_gb']])

WANT = {'lr': 7, 'rf': 8, 'hgb': 10}      # configurations per combination if time were unlimited
MIN = {'lr': 4, 'rf': 4, 'hgb': 6}
N_FOLDS = 5
t_one = pilot_tbl.set_index(['model', 'training'])['t_full_fit_min']
# The pilot times one anchor configuration. Average grid cost differs from the anchor, so scale by the grid:
# random forest cost grows with features tried per split; boosting cost grows with iterations; logistic regression
# is slower at weak penalties (assumed 2x). Notebook 04 compares projected with actual time on the first full fit.
ncols_rf = int(pilot_tbl.loc[pilot_tbl['model'] == 'rf', 'n_cols'].iloc[0])
frac = lambda f: (1 / np.sqrt(ncols_rf)) if f == 'sqrt' else f
MULT = dict(lr=2.0,
            rf=float(np.mean([frac(f) for f in SPACE['rf']['max_features']]) / frac(ANCHOR['rf']['max_features'])),
            hgb=float(np.mean(SPACE['hgb']['max_iter']) / ANCHOR['hgb']['max_iter']))
print('Grid cost multipliers relative to the anchor configuration:', {k: round(v, 2) for k, v in MULT.items()})
def projected_phases(n):
    # Phase 1: tune each model on the untreated fit (n configurations x 5 folds).
    tune = sum(n[k] * MULT[k] * N_FOLDS * t_one[(k, 'untreated')] for k in KINDS) / 60
    # Phase 2: refit the selected settings under weighting and oversampling (5 folds each). Cost taken at the grid average.
    other = sum(MULT[k] * N_FOLDS * t_one[(k, trt)] for k in KINDS for trt in ('weight', 'oversample')) / 60
    return tune, other
def projected_hours(n):
    return sum(projected_phases(n))
n = dict(WANT)
while projected_hours(n) > TIME_BUDGET_HOURS and any(n[k] > MIN[k] for k in KINDS):
    # drop one configuration from whichever model currently costs the most
    cost = {k: n[k] * MULT[k] * t_one[(k, 'untreated')] for k in KINDS if n[k] > MIN[k]}
    n[max(cost, key=cost.get)] -= 1
hours = projected_hours(n)
ph = projected_phases(n)
print('\nConfigurations per model (tuned on the untreated fit):', n)
print(f'Projected time: tuning {ph[0]:.1f} h + other two treatments {ph[1]:.1f} h = {hours:.1f} h (budget {TIME_BUDGET_HOURS} h)')
if hours > TIME_BUDGET_HOURS:
    print('WARNING: even the minimum search exceeds the budget. Raise TIME_BUDGET_HOURS, use High RAM, or reduce the grid.')
mem_ok = float(pilot_tbl['matrix_full_gb'].max()) < 0.4 * psutil.virtual_memory().total / 1e9
print('Largest encoded training matrix at full size (GB):', pilot_tbl['matrix_full_gb'].max(), '| fits comfortably in RAM:', mem_ok)

budget = dict(created=dt.datetime.now().isoformat(timespec='seconds'), seed=SEED, cores=N_JOBS, ram_gb=round(psutil.virtual_memory().total / 1e9, 1),
              sklearn=sklearn.__version__, scale_factor=round(scale, 3), time_exponent=EXP, time_budget_hours=TIME_BUDGET_HOURS,
              configs_per_model=n, tuned_on='untreated', phase_hours=[round(x, 2) for x in ph], cost_multipliers=MULT, projected_hours=round(hours, 2), min_configs=MIN, folds=N_FOLDS,
              rare_min_freq=RARE_MIN_FREQ, rf_trees=RF_TREES, anchor=ANCHOR, search_space=SPACE, memory_ok=bool(mem_ok))
(MOD / 'search_budget.json').write_text(json.dumps(budget, indent=2, default=str))
pilot_tbl.to_csv(MOD / 'pilot_timing.csv', index=False)
print('\nSaved:', *[p.name for p in sorted(MOD.glob('*'))], sep='\n - ')
print('\nNEXT: notebook 04 runs the grouped search with these budgets, saving each fold so a disconnect does not lose work.')

Scale factor: 5.02 (full training fold 208838 rows vs pilot 41595)


,model,training,t_full_fit_min,matrix_full_gb
0,lr,untreated,0.25,0.14
1,lr,weight,0.19,0.14
2,lr,oversample,0.35,0.21
3,rf,untreated,1.28,0.28
4,rf,weight,1.27,0.28
5,rf,oversample,2.02,0.44
6,hgb,untreated,3.16,0.28
7,hgb,weight,3.23,0.28
8,hgb,oversample,4.34,0.44


Grid cost multipliers relative to the anchor configuration: {'lr': 2.0, 'rf': 2.17, 'hgb': 1.38}

Configurations per model (tuned on the untreated fit): {'lr': 7, 'rf': 4, 'hgb': 6}
Projected time: tuning 3.4 h + other two treatments 1.6 h = 4.9 h (budget 5.0 h)
Largest encoded training matrix at full size (GB): 0.44 | fits comfortably in RAM: True

Saved:
 - folds_checksum.json
 - pilot_timing.csv
 - search_budget.json

NEXT: notebook 04 runs the grouped search with these budgets, saving each fold so a disconnect does not lose work.
